# Classical ML — Driver-Based Split

## 1. Load HOG Features

Load previously extracted HOG features and their corresponding labels.

In [5]:
import numpy as np
import pandas as pd
from pathlib import Path

# Locate project directory
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Load training metadata
train_df = pd.read_csv(
    ROOT / "data/splits/driver_based/train.csv"
)

# Load saved HOG features (64x48)
X_train = np.load(
    ROOT / "data/processed/driver_based/hog_64x48_train.npy"
)

# Extract labels and driver IDs
y_train = train_df["classname"].to_numpy()
groups_train = train_df["subject"].to_numpy()

# Verify matching number of samples
assert len(X_train) == len(y_train) == len(groups_train)

print("Features:", X_train.shape)
print("Labels:", y_train.shape)
print("Drivers:", len(np.unique(groups_train)))

Features: (15732, 1260)
Labels: (15732,)
Drivers: 18


## 2. Logistic Regression — Cross-Validation

Evaluate Logistic Regression using 4-fold StratifiedGroupKFold.

Drivers must not overlap between training and validation folds.

In [6]:
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Create 4-fold driver-independent cross-validation
cv = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

# Create preprocessing + classification pipeline
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000))
])

# Train and evaluate on each fold
scores = cross_validate(
    model,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring=["accuracy", "f1_macro"],
    n_jobs=1
)

# Display results
print("Accuracy per fold:", scores["test_accuracy"])
print("Macro-F1 per fold:", scores["test_f1_macro"])

print(f"\nMean Accuracy: {scores['test_accuracy'].mean():.4f}")
print(f"Accuracy Std: {scores['test_accuracy'].std():.4f}")

print(f"\nMean Macro-F1: {scores['test_f1_macro'].mean():.4f}")
print(f"Macro-F1 Std: {scores['test_f1_macro'].std():.4f}")

Accuracy per fold: [0.40660377 0.4363725  0.59825674 0.51720311]
Macro-F1 per fold: [0.4266527  0.44388998 0.58989271 0.52773054]

Mean Accuracy: 0.4896
Accuracy Std: 0.0746

Mean Macro-F1: 0.4970
Macro-F1 Std: 0.0658


In [7]:
# Save cross-validation results
results_dir = ROOT / "results/classical/driver_based"
results_dir.mkdir(parents=True, exist_ok=True)

results_df = pd.DataFrame({
    "model": "Logistic Regression",
    "resolution": "64x48",
    "fold": range(1, 5),
    "accuracy": scores["test_accuracy"],
    "macro_f1": scores["test_f1_macro"]
})

results_df.to_csv(
    results_dir / "logistic_regression_64x48.csv",
    index=False
)

print(results_df)

                 model resolution  fold  accuracy  macro_f1
0  Logistic Regression      64x48     1  0.406604  0.426653
1  Logistic Regression      64x48     2  0.436373  0.443890
2  Logistic Regression      64x48     3  0.598257  0.589893
3  Logistic Regression      64x48     4  0.517203  0.527731


## 3. Support Vector Machine (SVM) — Cross-Validation

Evaluate a linear SVM using HOG features and 4-fold
driver-independent cross-validation.

In [8]:
from sklearn.svm import LinearSVC

# Create SVM pipeline with feature scaling
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(C=1.0, max_iter=5000, random_state=42))
])

# Evaluate using the same cross-validation folds
svm_scores = cross_validate(
    svm_model,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring=["accuracy", "f1_macro"],
    n_jobs=1
)

# Display results
print("Accuracy per fold:", svm_scores["test_accuracy"])
print("Macro-F1 per fold:", svm_scores["test_f1_macro"])

print(f"\nMean Accuracy: {svm_scores['test_accuracy'].mean():.4f}")
print(f"Accuracy Std: {svm_scores['test_accuracy'].std():.4f}")

print(f"\nMean Macro-F1: {svm_scores['test_f1_macro'].mean():.4f}")
print(f"Macro-F1 Std: {svm_scores['test_f1_macro'].std():.4f}")



Accuracy per fold: [0.39056604 0.38347148 0.52139461 0.45754717]
Macro-F1 per fold: [0.38964779 0.38435511 0.49864697 0.46050837]

Mean Accuracy: 0.4382
Accuracy Std: 0.0560

Mean Macro-F1: 0.4333
Macro-F1 Std: 0.0482


In [9]:
# Save SVM cross-validation results
svm_results = pd.DataFrame({
    "model": "Linear SVM",
    "resolution": "64x48",
    "fold": range(1, 5),
    "accuracy": svm_scores["test_accuracy"],
    "macro_f1": svm_scores["test_f1_macro"],
    "fit_time": svm_scores["fit_time"]
})

svm_results.to_csv(
    results_dir / "linear_svm_64x48.csv",
    index=False
)

print(svm_results)

        model resolution  fold  accuracy  macro_f1    fit_time
0  Linear SVM      64x48     1  0.390566  0.389648  188.063229
1  Linear SVM      64x48     2  0.383471  0.384355  197.629248
2  Linear SVM      64x48     3  0.521395  0.498647  199.263910
3  Linear SVM      64x48     4  0.457547  0.460508  210.143842


## 4. Random Forest — Cross-Validation

Evaluate Random Forest using HOG features and 4-fold
driver-independent cross-validation.

In [11]:
from sklearn.ensemble import RandomForestClassifier

# Create Random Forest with 100 decision trees
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

# Evaluate using the same driver-independent folds
rf_scores = cross_validate(
    rf_model,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring=["accuracy", "f1_macro"],
    n_jobs=1
)

# Display cross-validation results
print("Accuracy per fold:", rf_scores["test_accuracy"])
print("Macro-F1 per fold:", rf_scores["test_f1_macro"])

print(f"\nMean Accuracy: {rf_scores['test_accuracy'].mean():.4f}")
print(f"Accuracy Std: {rf_scores['test_accuracy'].std():.4f}")

print(f"\nMean Macro-F1: {rf_scores['test_f1_macro'].mean():.4f}")
print(f"Macro-F1 Std: {rf_scores['test_f1_macro'].std():.4f}")

Accuracy per fold: [0.38985849 0.42881521 0.59852087 0.5463374 ]
Macro-F1 per fold: [0.38075898 0.41202571 0.58290448 0.53615265]

Mean Accuracy: 0.4909
Accuracy Std: 0.0847

Mean Macro-F1: 0.4780
Macro-F1 Std: 0.0840


In [12]:
# Check training time for each fold
print("Training times:", rf_scores["fit_time"])

# Check validation scoring time
print("Scoring times:", rf_scores["score_time"])

# Check if any folds produced invalid results
print("Accuracy scores:", rf_scores["test_accuracy"])

Training times: [0.98310685 0.95037127 0.98350501 1.01869798]
Scoring times: [0.02952719 0.02956223 0.01895189 0.02872872]
Accuracy scores: [0.38985849 0.42881521 0.59852087 0.5463374 ]


In [13]:
# Create results directory
results_dir = ROOT / "results/classical/driver_based"
results_dir.mkdir(parents=True, exist_ok=True)

# Organize results from all 4 folds
rf_results = pd.DataFrame({
    "model": "Random Forest",
    "resolution": "64x48",
    "fold": range(1, 5),
    "accuracy": rf_scores["test_accuracy"],
    "macro_f1": rf_scores["test_f1_macro"],
    "fit_time": rf_scores["fit_time"]
})

# Save results permanently
rf_results.to_csv(
    results_dir / "random_forest_64x48.csv",
    index=False
)

print(rf_results)

           model resolution  fold  accuracy  macro_f1  fit_time
0  Random Forest      64x48     1  0.389858  0.380759  0.983107
1  Random Forest      64x48     2  0.428815  0.412026  0.950371
2  Random Forest      64x48     3  0.598521  0.582904  0.983505
3  Random Forest      64x48     4  0.546337  0.536153  1.018698


## 5. K-Nearest Neighbors (KNN) — Cross-Validation

Evaluate KNN using HOG features and 4-fold
driver-independent cross-validation.

In [14]:
from sklearn.neighbors import KNeighborsClassifier

# Create KNN pipeline with feature scaling
knn_model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=5,
        weights="uniform",
        n_jobs=-1
    ))
])

# Evaluate using the same driver-independent folds
knn_scores = cross_validate(
    knn_model,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring=["accuracy", "f1_macro"],
    n_jobs=1
)

# Display cross-validation results
print("Accuracy per fold:", knn_scores["test_accuracy"])
print("Macro-F1 per fold:", knn_scores["test_f1_macro"])

print(f"\nMean Accuracy: {knn_scores['test_accuracy'].mean():.4f}")
print(f"Accuracy Std: {knn_scores['test_accuracy'].std():.4f}")

print(f"\nMean Macro-F1: {knn_scores['test_f1_macro'].mean():.4f}")
print(f"Macro-F1 Std: {knn_scores['test_f1_macro'].std():.4f}")

Accuracy per fold: [0.34716981 0.39980497 0.54120444 0.49556049]
Macro-F1 per fold: [0.33596405 0.38707672 0.53285261 0.48806661]

Mean Accuracy: 0.4459
Accuracy Std: 0.0765

Mean Macro-F1: 0.4360
Macro-F1 Std: 0.0782


In [15]:
# Save KNN cross-validation results
knn_results = pd.DataFrame({
    "model": "KNN",
    "resolution": "64x48",
    "fold": range(1, 5),
    "accuracy": knn_scores["test_accuracy"],
    "macro_f1": knn_scores["test_f1_macro"],
    "fit_time": knn_scores["fit_time"],
    "score_time": knn_scores["score_time"]
})

knn_results.to_csv(
    results_dir / "knn_64x48.csv",
    index=False
)

print(knn_results)

  model resolution  fold  accuracy  macro_f1  fit_time  score_time
0   KNN      64x48     1  0.347170  0.335964  0.070482    0.285926
1   KNN      64x48     2  0.399805  0.387077  0.065658    0.234601
2   KNN      64x48     3  0.541204  0.532853  0.066580    0.213542
3   KNN      64x48     4  0.495560  0.488067  0.065749    0.219081


## 6. HOG Resolution Comparison

Investigate how image resolution affects classification performance.

Resolutions:
- 64×48
- 128×96
- 256×192

Use the same model and cross-validation folds for fair comparison.

# Classical Machine Learning — Driver-Based Split

## 1. HOG Features: 128×96

Load and verify the training features and labels.

Each image is represented by 5940 HOG features.
The final test set remains untouched.

In [5]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("..")

# Load training metadata
train_df = pd.read_csv(ROOT / "data/splits/driver_based/train.csv")

# Load ONLY 128x96 HOG features
X_train = np.load(
    ROOT / "data/processed/driver_based/hog_128x96_train.npy",
    mmap_mode="r"
)

# Load labels and driver IDs
y_train = train_df["classname"].to_numpy()
groups_train = train_df["subject"].to_numpy()

# Check dimensions
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("Unique drivers:", len(np.unique(groups_train)))
print("Classes:", np.unique(y_train))

X_train: (15732, 5940)
y_train: (15732,)
Unique drivers: 18
Classes: ['c0' 'c1' 'c2' 'c3' 'c4' 'c5' 'c6' 'c7' 'c8' 'c9']


## 2. Logistic Regression — 4-Fold Cross-Validation

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, StratifiedGroupKFold

cv = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

# Create Logistic Regression pipeline
lr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000))
])


# Train and evaluate using existing 4-fold CV
lr_scores = cross_validate(
    lr_model,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring=["accuracy", "f1_macro"],
    n_jobs=1
)

# Display results for all 4 folds
print("Accuracy per fold:", lr_scores["test_accuracy"])
print("Macro-F1 per fold:", lr_scores["test_f1_macro"])

print(f"\nMean Accuracy: {lr_scores['test_accuracy'].mean():.4f}")
print(f"Accuracy Std: {lr_scores['test_accuracy'].std():.4f}")

print(f"\nMean Macro-F1: {lr_scores['test_f1_macro'].mean():.4f}")
print(f"Macro-F1 Std: {lr_scores['test_f1_macro'].std():.4f}")

Accuracy per fold: [0.41014151 0.58069235 0.76069731 0.69034406]
Macro-F1 per fold: [0.4255203  0.57923797 0.75273559 0.68602231]

Mean Accuracy: 0.6105
Accuracy Std: 0.1323

Mean Macro-F1: 0.6109
Macro-F1 Std: 0.1236


In [7]:
# Create results directory
results_dir = ROOT / "results/classical/driver_based"
results_dir.mkdir(parents=True, exist_ok=True)

# Organize results from all 4 folds
lr_results = pd.DataFrame({
    "model": "Logistic Regression",
    "resolution": "128x96",
    "fold": range(1, 5),
    "accuracy": lr_scores["test_accuracy"],
    "macro_f1": lr_scores["test_f1_macro"],
    "fit_time": lr_scores["fit_time"]
})

# Save results
lr_results.to_csv(
    results_dir / "logistic_regression_128x96.csv",
    index=False
)

print(lr_results)

                 model resolution  fold  accuracy  macro_f1  fit_time
0  Logistic Regression     128x96     1  0.410142  0.425520  3.298761
1  Logistic Regression     128x96     2  0.580692  0.579238  3.929901
2  Logistic Regression     128x96     3  0.760697  0.752736  3.775546
3  Logistic Regression     128x96     4  0.690344  0.686022  3.634526


## 3. Linear SVM — 4-Fold Cross-Validation

In [ ]:
from sklearn.svm import LinearSVC

# Create SVM pipeline with feature scaling
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(C=1.0, max_iter=5000, random_state=42))
])

# Run all 4 folds in parallel
svm_scores = cross_validate(
    svm_model,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring=["accuracy", "f1_macro"],
    n_jobs=4,
    verbose=2
)

# Display results for all 4 folds
print("Accuracy per fold:", svm_scores["test_accuracy"])
print("Macro-F1 per fold:", svm_scores["test_f1_macro"])

print(f"\nMean Accuracy: {svm_scores['test_accuracy'].mean():.4f}")
print(f"Accuracy Std: {svm_scores['test_accuracy'].std():.4f}")

print(f"\nMean Macro-F1: {svm_scores['test_f1_macro'].mean():.4f}")
print(f"Macro-F1 Std: {svm_scores['test_f1_macro'].std():.4f}")

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.


In [ ]:
import pandas as pd
from pathlib import Path
from joblib import dump
from sklearn.base import clone

# 1. Save cross-validation results
results_dir = ROOT / "results/classical/driver_based"
results_dir.mkdir(parents=True, exist_ok=True)

svm_results = pd.DataFrame({
    "model": "Linear SVM",
    "resolution": "128x96",
    "fold": range(1, 5),
    "accuracy": svm_scores["test_accuracy"],
    "macro_f1": svm_scores["test_f1_macro"],
    "fit_time": svm_scores["fit_time"]
})

results_path = results_dir / "linear_svm_128x96.csv"
svm_results.to_csv(results_path, index=False)

print(f"Cross-validation results saved: {results_path}", flush=True)

# 2. Train the final model on all training images
final_svm = clone(svm_model)
final_svm.fit(X_train, y_train)

# 3. Save the trained model
model_dir = ROOT / "models/classical/driver_based"
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "linear_svm_128x96.joblib"
dump(final_svm, model_path)

print(f"Trained SVM model saved: {model_path}", flush=True)